In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans

In [2]:
def load_and_preprocess(path: str) -> pd.DataFrame:
    """
    Load CSV and engineer only features that are DERIVED from existing columns.
    No new columns are invented — completion_rate and overdue_rate are
    direct mathematical transformations of existing columns.
    """
    df = pd.read_csv(path)

    # Derived ratios (pure math on existing columns — not new data)
    df["completion_rate"] = df["tasks_completed"] / df["tasks_assigned"].replace(0, np.nan)
    df["overdue_rate"]    = df["overdue_tasks"]   / df["tasks_assigned"].replace(0, np.nan)
    df[["completion_rate", "overdue_rate"]] = df[["completion_rate", "overdue_rate"]].fillna(0)
    df[["completion_rate", "overdue_rate"]] = df[["completion_rate", "overdue_rate"]].clip(0, 1)

    return df


df = load_and_preprocess("C:\\Users\\AIA\\Downloads\\teamify_dataset.csv")
print(f" Data loaded: {df.shape[0]} users, {df.shape[1]} columns\n")

 Data loaded: 5000 users, 14 columns



In [3]:
"""
Profile is built from the columns that best represent a user's work profile:
  · Skill proxy      → skill_match_score
  · Experience proxy → avg_rating, tasks_assigned
  · Output proxy     → completion_rate, overdue_rate, quality_score
  · Collaboration    → teamwork_score, attendance_rate, availability_score
"""

PROFILE_COLS = [
    "skill_match_score",       # skill fit
    "avg_rating",              # historical performance
    "tasks_assigned",          # workload (experience proxy)
    "completion_rate",         # reliability
    "overdue_rate",            # punctuality (negative)
    "quality_score",           # work quality
    "teamwork_score",          # collaboration
    "attendance_rate",         # commitment
    "availability_score",      # future availability
]

def build_profile(user_row: pd.Series) -> dict:
    """Return a human-readable profile dict for a single user row."""
    return {
        "skill_fit"         : round(float(user_row["skill_match_score"]), 2),
        "historical_rating" : round(float(user_row["avg_rating"]), 2),
        "tasks_assigned"    : int(user_row["tasks_assigned"]),
        "completion_rate"   : f"{user_row['completion_rate']*100:.1f}%",
        "overdue_rate"      : f"{user_row['overdue_rate']*100:.1f}%",
        "quality_score"     : round(float(user_row["quality_score"]), 2),
        "teamwork_score"    : round(float(user_row["teamwork_score"]), 2),
        "attendance_rate"   : f"{user_row['attendance_rate']*100:.1f}%",
        "availability"      : round(float(user_row["availability_score"]), 2),
    }



In [5]:
"""
Target  → final_rating  (continuous, range 1.73 – 4.80)
Features→ PROFILE_COLS  (all numeric, derived only from existing columns)
Model   → GradientBoostingRegressor (best balance of accuracy & interpretability
          for tabular regression; no deep learning overhead needed)
"""

FEATURE_COLS = PROFILE_COLS  # reuse profile columns as model features

scaler  = MinMaxScaler()
X_raw   = df[FEATURE_COLS].values
y       = df["final_rating"].values
X_scaled = scaler.fit_transform(X_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

rating_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.08,
    max_depth=4,
    random_state=42
)
rating_model.fit(X_train, y_train)
y_pred = rating_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)
print(f" Rating Model Evaluation")
print(f"   MAE  : {mae:.4f}  (avg error in rating points)")
print(f"   R²   : {r2:.4f}  (variance explained)\n")


def predict_rating(user_data: dict) -> dict:
    """
    Predict the performance rating for a user.

    Args:
        user_data: dict with keys matching PROFILE_COLS.
                   'completion_rate' and 'overdue_rate' must be provided
                   as floats in [0, 1].

    Returns:
        dict with 'predicted_rating' (float) and 'performance_label' (str).

    Example:
        predict_rating({
            "skill_match_score": 0.75,
            "avg_rating": 4.2,
            "tasks_assigned": 15,
            "completion_rate": 0.80,
            "overdue_rate": 0.10,
            "quality_score": 4.0,
            "teamwork_score": 3.8,
            "attendance_rate": 0.95,
            "availability_score": 0.70,
        })
    """
    row = pd.DataFrame([user_data])[FEATURE_COLS]
    row_scaled = scaler.transform(row.values)
    rating = float(rating_model.predict(row_scaled)[0])
    rating = round(np.clip(rating, 1.0, 5.0), 2)

    if   rating >= 4.0: label = " Excellent"
    elif rating >= 3.0: label = " Good"
    elif rating >= 2.0: label = "  Needs Improvement"
    else:               label = " Poor"

    return {"predicted_rating": rating, "performance_label": label}


 Rating Model Evaluation
   MAE  : 0.0282  (avg error in rating points)
   R²   : 0.9959  (variance explained)



In [6]:
"""
Available text column: feedback_text
Only 6 distinct phrases exist in the dataset, so a keyword/rule-based
sentiment system is more reliable than a trained model trained on 5000
near-identical strings (which would just memorize).

Approach:
  1. Keyword matching on known phrases (high precision on this dataset)
  2. Polarity fallback via simple positive/negative word lists

Output: sentiment (Positive / Negative / Neutral) + confidence score
"""

POSITIVE_KEYWORDS = {
    "excellent", "outstanding", "reliable", "committed",
    "good", "great", "consistent"
}
NEGATIVE_KEYWORDS = {
    "late", "weak", "needs improvement", "not consistent",
    "poor", "bad", "overdue"
}

PHRASE_MAP = {
    "excellent performance and teamwork" : ("Positive", 0.97),
    "outstanding contribution"           : ("Positive", 0.96),
    "very committed and reliable"        : ("Positive", 0.94),
    "good but not consistent"            : ("Neutral",  0.72),
    "needs improvement in deadlines"     : ("Negative", 0.88),
    "late submission and weak teamwork"  : ("Negative", 0.95),
}


def analyze_feedback(text: str) -> dict:
    """
    Perform sentiment analysis on a feedback string.

    Args:
        text: raw feedback text from 'feedback_text' column or free input.

    Returns:
        dict with 'sentiment' (Positive/Negative/Neutral),
        'confidence' (float 0–1), and 'keywords_found' (list).

    Example:
        analyze_feedback("Excellent performance and teamwork")
    """
    cleaned = text.strip().lower()

    # 1. Exact phrase matching (covers 100% of dataset entries)
    if cleaned in PHRASE_MAP:
        sentiment, confidence = PHRASE_MAP[cleaned]
        return {
            "original_text" : text,
            "sentiment"     : sentiment,
            "confidence"    : confidence,
            "keywords_found": [cleaned],
            "method"        : "phrase_match",
        }

    # 2. Keyword scan for unseen text
    words     = set(cleaned.split())
    pos_hits  = POSITIVE_KEYWORDS & words
    neg_hits  = NEGATIVE_KEYWORDS & words
    # Also check multi-word negatives
    for kw in NEGATIVE_KEYWORDS:
        if kw in cleaned:
            neg_hits.add(kw)

    if pos_hits and not neg_hits:
        sentiment, confidence = "Positive", 0.75
    elif neg_hits and not pos_hits:
        sentiment, confidence = "Negative", 0.75
    elif pos_hits and neg_hits:
        sentiment, confidence = "Neutral",  0.60
    else:
        sentiment, confidence = "Neutral",  0.50

    return {
        "original_text" : text,
        "sentiment"     : sentiment,
        "confidence"    : confidence,
        "keywords_found": list(pos_hits | neg_hits),
        "method"        : "keyword_scan",
    }

In [7]:
"""
Approach: Cosine Similarity on normalised profile features.

Why cosine similarity?
  · Finds users with similar overall work-profile shape.
  · Unaffected by scale differences across features.
  · Fast and interpretable.

Optional: KMeans clustering is also shown to group users into
performance tiers (for a "find teammates in same performance class"
use case), but the primary function uses cosine similarity.

Features used for matching:
  · PROFILE_COLS (same as rating model — all numeric, existing columns)
  · project_similarity is included to match users with similar project
    contexts (it is an existing column in the dataset).
"""

MATCH_COLS = PROFILE_COLS + ["project_similarity"]

match_scaler = MinMaxScaler()
X_match      = match_scaler.fit_transform(df[MATCH_COLS].values)
sim_matrix   = cosine_similarity(X_match)   # (5000 × 5000)

# ── Optional: cluster users into performance tiers ──
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_match)
CLUSTER_LABELS = {0: "High Performer", 1: "Consistent Worker",
                  2: "Developing",     3: "Struggling"}


def recommend_team(user_id: int, top_n: int = 5) -> dict:
    """
    Recommend the best teammate candidates for a given user.

    Args:
        user_id : integer index of the user in the dataset (0-based).
        top_n   : number of recommendations to return (default 5).

    Returns:
        dict with:
          · user_profile     – profile of the queried user
          · recommendations  – list of dicts (id, similarity, profile, cluster)

    Example:
        recommend_team(user_id=42, top_n=5)
    """
    if user_id < 0 or user_id >= len(df):
        raise ValueError(f"user_id must be between 0 and {len(df)-1}")

    sims        = sim_matrix[user_id].copy()
    sims[user_id] = -1                     # exclude self
    top_indices = np.argsort(sims)[::-1][:top_n]

    user_row    = df.iloc[user_id]
    user_profile = build_profile(user_row)
    user_profile["cluster"] = CLUSTER_LABELS.get(int(user_row["cluster"]), "Unknown")

    recs = []
    for idx in top_indices:
        rec_row = df.iloc[idx]
        recs.append({
            "candidate_id"  : int(idx),
            "similarity"    : round(float(sims[idx]), 4),
            "cluster"       : CLUSTER_LABELS.get(int(rec_row["cluster"]), "Unknown"),
            "profile"       : build_profile(rec_row),
            "predicted_rating": predict_rating({col: rec_row[col] for col in FEATURE_COLS})["predicted_rating"],
        })

    return {
        "query_user_id"  : user_id,
        "user_profile"   : user_profile,
        "recommendations": recs,
    }

In [8]:
if __name__ == "__main__":
    print("=" * 60)
    print("DEMO: predict_rating()")
    print("=" * 60)
    sample_user = {
        "skill_match_score" : 0.75,
        "avg_rating"        : 4.1,
        "tasks_assigned"    : 15,
        "completion_rate"   : 0.80,
        "overdue_rate"      : 0.07,
        "quality_score"     : 3.9,
        "teamwork_score"    : 4.2,
        "attendance_rate"   : 0.93,
        "availability_score": 0.68,
    }
    result = predict_rating(sample_user)
    print(f"  Input         : {sample_user}")
    print(f"  Predicted     : {result['predicted_rating']} — {result['performance_label']}")

    print()
    print("=" * 60)
    print("DEMO: analyze_feedback()")
    print("=" * 60)
    test_feedbacks = [
        "Excellent performance and teamwork",
        "Late submission and weak teamwork",
        "Good but not consistent",
        "Very committed and reliable",
        "Shows outstanding effort but misses deadlines",  # unseen text
    ]
    for txt in test_feedbacks:
        res = analyze_feedback(txt)
        print(f"  [{res['sentiment']:8s} | {res['confidence']:.2f}]  \"{txt}\"")

    print()
    print("=" * 60)
    print("DEMO: recommend_team(user_id=0)")
    print("=" * 60)
    team_result = recommend_team(user_id=0, top_n=3)
    print(f"  Query user profile : {team_result['user_profile']}")
    print(f"\n  Top 3 Recommendations:")
    for i, rec in enumerate(team_result["recommendations"], 1):
        print(f"\n  [{i}] Candidate ID  : {rec['candidate_id']}")
        print(f"       Similarity    : {rec['similarity']}")
        print(f"       Cluster       : {rec['cluster']}")
        print(f"       Pred. Rating  : {rec['predicted_rating']}")
        print(f"       Profile       : {rec['profile']}")

DEMO: predict_rating()
  Input         : {'skill_match_score': 0.75, 'avg_rating': 4.1, 'tasks_assigned': 15, 'completion_rate': 0.8, 'overdue_rate': 0.07, 'quality_score': 3.9, 'teamwork_score': 4.2, 'attendance_rate': 0.93, 'availability_score': 0.68}
  Predicted     : 4.06 —  Excellent

DEMO: analyze_feedback()
  [Positive | 0.97]  "Excellent performance and teamwork"
  [Negative | 0.95]  "Late submission and weak teamwork"
  [Neutral  | 0.72]  "Good but not consistent"
  [Positive | 0.94]  "Very committed and reliable"
  [Positive | 0.75]  "Shows outstanding effort but misses deadlines"

DEMO: recommend_team(user_id=0)
  Query user profile : {'skill_fit': 0.72, 'historical_rating': 4.9, 'tasks_assigned': 16, 'completion_rate': '75.0%', 'overdue_rate': '12.5%', 'quality_score': 3.9, 'teamwork_score': 4.5, 'attendance_rate': '100.0%', 'availability': 0.65, 'cluster': 'Consistent Worker'}

  Top 3 Recommendations:

  [1] Candidate ID  : 43
       Similarity    : 0.9857
       Cluster 